# Survey Data Cleaning
Created: June 30th, 2025

Created by: Juli Hirt

## Imports

In [ ]:
#imports
import pandas as pd
import numpy as np
import os
from spellchecker import SpellChecker
import json

In [ ]:

cwd = os.getcwd()
print(cwd)

## Initial Load

In [ ]:
survey_data = 'Video Game Mood Taxonomy Survey - Final_June 17, 2025_15.56.xlsx'
df = pd.read_excel(f'Data/Inbound/{survey_data}', sheet_name='Sheet0')
df.head(3)

In [ ]:
question_df = df[df['StartDate']== 'Start Date']
question_df.head(3)

In [ ]:
response_df = df[(df['StartDate']!= 'Start Date') & (df['Status']!= '{"ImportId":"status"}')].reset_index(drop=True)
response_df.head(3)

In [ ]:
response_df.shape

## Establish DataFrame and Metadata

In [ ]:
# create new dataframe
cleaned_df = pd.DataFrame()

## Clean Question Q2 (Recent Mood)

In [ ]:
recent_q = ['Q2_1','Q2_2','Q2_3','Q2_4','Q2_5','Q2_6','Q2_7','Q2_8','Q2_9','Q2_10']
recent_categories = ['Q3_1','Q3_2','Q3_3','Q3_4','Q3_5','Q3_6','Q3_7','Q3_8','Q3_9','Q3_10']

In [ ]:
recent_q_terms = []
term_template = {
    'metadata.ResponseId': '',
    'metadata.Finished': '',
    'metadata.Progress': '',
    'metadata.TermTrack': '',
    'metadata.QuestionId': '',
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
    'QX_X.UserCategoryMapping': '',
    'QX.MultipleCategoryTerm': '',
    'QX#1_X.MultipleCategoryMapping': '',
    'QX#2_X_1.UserProvidedCategory': '',
    'QX#2_X_1.CleanedCategory': '',
}

In [ ]:
for idx, row in response_df.iterrows():
    # if the progress < 33, skip the row. They have not completed all of the modules for recent game mood terms
    if row['Progress'] < 33:
        continue
    try:
        for id,q in enumerate(recent_q):
            if pd.isna(row[q]):
                continue
            else:
                json_obj = term_template.copy()
                json_obj['metadata.TermTrack'] = 'Recent'
                json_obj['metadata.ResponseId'] = row['ResponseId']
                json_obj['metadata.Finished'] = row['Finished']
                json_obj['metadata.Progress'] = row['Progress']
                json_obj['metadata.QuestionId'] = q
                question_id = q.split('_')[1]

                multiple_categories = row['Q4']
                if multiple_categories != 'No':
                    multiple_category_terms = multiple_categories.split(',')
                    multiple_category_terms = [cat.strip('Mood Term') for cat in multiple_category_terms]   
                else: 
                    multiple_category_terms = []    

                #check if the mood term is NaN
                
                json_obj['QX_X.UserProvidedMoodTerm'] = row[q]

                #clean the mood term
                #step 1 - remove leading hashtag
                mood_term = row[q].lstrip('#').strip()
                #step 2 - lowercase
                mood_term = mood_term.lower()
                #step 3 - spell check
                spell = SpellChecker()
                mood_term_cleaned = spell.correction(mood_term)

                if mood_term_cleaned != None:
                    json_obj['QX_X.CleanedMoodTerm'] = mood_term_cleaned
                else: json_obj['QX_X.CleanedMoodTerm'] = mood_term
                    
                #gather the user mapped category
                json_obj['QX_X.UserCategoryMapping'] = row[recent_categories[id]]

                #identify multiple categories
                if question_id in multiple_category_terms:
                    json_obj['QX.MultipleCategoryTerm'] = True

                    if pd.notna(row[f'Q5#1_{int(question_id)+1}']):
                        #split the multiple categories by comma
                        json_obj['QX#1_X.MultipleCategoryMapping'] = row[f'Q5#1_{int(question_id)+1}'].split(',')
                    else: 
                        json_obj['QX#1_X.MultipleCategoryMapping'] = None

                    if pd.notna(row[f'Q5#2_{int(question_id)+1}_1']):
                        json_obj['QX#2_X_1.UserProvidedCategory'] = row[f'Q5#2_{int(question_id)+1}_1']
                        #clean the category
                        user_provided_category = row[f'Q5#2_{int(question_id)+1}_1'].strip().lower()
                        user_provided_category_cleaned = spell.correction(user_provided_category)

                        if mood_term_cleaned != None:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category_cleaned
                        else:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category
                    else:
                        json_obj['QX#2_X_1.UserProvidedCategory'] = None
                        json_obj['QX#2_X_1.CleanedCategory'] = None

                else:
                    json_obj['QX.MultipleCategoryTerm'] = False
                    json_obj['QX#1_X.MultipleCategoryMapping'] = None
                    json_obj['QX#2_X_1.UserProvidedCategory'] = None
                    json_obj['QX#2_X_1.CleanedCategory'] = None

                recent_q_terms.append(json_obj)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        break

In [ ]:
# testing for a specific row
response_df.loc[66]['ResponseId']

In [ ]:
# testing print of data
print(json.dumps(recent_q_terms, indent=2))

In [ ]:
# convert the array to a DataFrame and save to CSV
recent_q_df = pd.DataFrame(recent_q_terms)
recent_q_df.to_csv('Data/Outbound/recent_q_terms.csv', index=False, header=True)

In [ ]:
recent_q_df.shape

In [ ]:
#test to ensure that all terms are captured
recent_terms_df = response_df[response_df['Progress'] >= 33][recent_q].copy()
total_recent_terms = pd.DataFrame(recent_terms_df[recent_q].count(axis=1))
total_recent_terms.rename(columns={0:'Total_Recent_Terms'}, inplace=True)
total_recent_terms.sum()

## Clean Question Q6 (Favorite Mood)

In [ ]:
fav_q = ['Q6_1','Q6_2','Q6_3','Q6_4','Q6_5','Q6_6','Q6_7','Q6_8','Q6_9','Q6_10']
fav_categories = ['Q7_1','Q7_2','Q7_3','Q7_4','Q7_5','Q7_6','Q7_7','Q7_8','Q7_9','Q7_10']

In [ ]:
fav_q_terms = []
term_template = {
    'metadata.ResponseId': '',
    'metadata.Finished': '',
    'metadata.Progress': '',
    'metadata.TermTrack': '',
    'metadata.QuestionId': '',
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
    'QX_X.UserCategoryMapping': '',
    'QX.MultipleCategoryTerm': '',
    'QX#1_X.MultipleCategoryMapping': '',
    'QX#2_X_1.UserProvidedCategory': '',
    'QX#2_X_1.CleanedCategory': '',
}

In [ ]:
for idx, row in response_df.iterrows():
    # if the progress < 57, skip the row. They have not completed all of the modules for fav game mood terms
    if row['Progress'] < 57:
        continue
    try:
        for id,q in enumerate(fav_q):
            if pd.isna(row[q]):
                continue
            else:
                json_obj = term_template.copy()
                json_obj['metadata.TermTrack'] = 'Favorite'
                json_obj['metadata.ResponseId'] = row['ResponseId']
                json_obj['metadata.Finished'] = row['Finished']
                json_obj['metadata.Progress'] = row['Progress']
                json_obj['metadata.QuestionId'] = q
                question_id = q.split('_')[1]

                multiple_categories = row['Q8']
                if multiple_categories != 'No':
                    multiple_category_terms = multiple_categories.split(',')
                    multiple_category_terms = [cat.strip('Mood Term') for cat in multiple_category_terms]   
                else: 
                    multiple_category_terms = []    

                #check if the mood term is NaN
                
                json_obj['QX_X.UserProvidedMoodTerm'] = row[q]

                #clean the mood term
                #step 1 - remove leading hashtag
                mood_term = row[q].lstrip('#').strip()
                #step 2 - lowercase
                mood_term = mood_term.lower()
                #step 3 - spell check
                spell = SpellChecker()
                mood_term_cleaned = spell.correction(mood_term)

                if mood_term_cleaned != None:
                    json_obj['QX_X.CleanedMoodTerm'] = mood_term_cleaned
                else: json_obj['QX_X.CleanedMoodTerm'] = mood_term
                    
                #gather the user mapped category
                json_obj['QX_X.UserCategoryMapping'] = row[fav_categories[id]]

                #identify multiple categories
                if question_id in multiple_category_terms:
                    json_obj['QX.MultipleCategoryTerm'] = True

                    if pd.notna(row[f'Q9#1_{int(question_id)+1}']):
                        #split the multiple categories by comma
                        json_obj['QX#1_X.MultipleCategoryMapping'] = row[f'Q9#1_{int(question_id)+1}'].split(',')
                    else: 
                        json_obj['QX#1_X.MultipleCategoryMapping'] = None

                    if pd.notna(row[f'Q9#2_{int(question_id)+1}_1']):
                        json_obj['QX#2_X_1.UserProvidedCategory'] = row[f'Q9#2_{int(question_id)+1}_1']
                        #clean the category
                        user_provided_category = row[f'Q9#2_{int(question_id)+1}_1'].strip().lower()
                        user_provided_category_cleaned = spell.correction(user_provided_category)

                        if mood_term_cleaned != None:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category_cleaned
                        else:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category
                    else:
                        json_obj['QX#2_X_1.UserProvidedCategory'] = None
                        json_obj['QX#2_X_1.CleanedCategory'] = None

                else:
                    json_obj['QX.MultipleCategoryTerm'] = False
                    json_obj['QX#1_X.MultipleCategoryMapping'] = None
                    json_obj['QX#2_X_1.UserProvidedCategory'] = None
                    json_obj['QX#2_X_1.CleanedCategory'] = None

                fav_q_terms.append(json_obj)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        break

In [ ]:
# convert the array to a DataFrame and save to CSV
fav_q_df = pd.DataFrame(fav_q_terms)
fav_q_df.to_csv('Data/Outbound/fav_q_terms.csv', index=False, header=True)

In [ ]:
fav_q_df.shape

In [ ]:
#test to ensure that all terms are captured
fav_terms_df = response_df[response_df['Progress'] >= 57][fav_q].copy()
total_fav_terms = pd.DataFrame(fav_terms_df[fav_q].count(axis=1))
total_fav_terms.rename(columns={0:'Total_Favorite_Terms'}, inplace=True)
total_fav_terms.sum()

## Clean Question Q10 (Avoid Mood)

In [ ]:
avoid_q = ['Q10_1','Q10_2','Q10_3','Q10_4','Q10_5','Q10_6','Q10_7','Q10_8','Q10_9','Q10_10']
avoid_categories = ['Q11_1','Q11_2','Q11_3','Q11_4','Q11_5','Q11_6','Q11_7','Q11_8','Q11_9','Q11_10']

In [ ]:
avoid_q_terms = []
term_template = {
    'metadata.ResponseId': '',
    'metadata.Finished': '',
    'metadata.Progress': '',
    'metadata.TermTrack': '',
    'metadata.QuestionId': '',
    'QX_X.UserProvidedMoodTerm': '',
    'QX_X.CleanedMoodTerm': '',
    'QX_X.UserCategoryMapping': '',
    'QX.MultipleCategoryTerm': '',
    'QX#1_X.MultipleCategoryMapping': '',
    'QX#2_X_1.UserProvidedCategory': '',
    'QX#2_X_1.CleanedCategory': '',
}

In [ ]:
for idx, row in response_df.iterrows():
    # if the progress < 100, skip the row. They have not completed all of the modules for fav game mood terms
    if row['Progress'] < 100:
        continue
    try:
        for id,q in enumerate(avoid_q):
            if pd.isna(row[q]):
                continue
            else:
                json_obj = term_template.copy()
                json_obj['metadata.TermTrack'] = 'Avoid'
                json_obj['metadata.ResponseId'] = row['ResponseId']
                json_obj['metadata.Finished'] = row['Finished']
                json_obj['metadata.Progress'] = row['Progress']
                json_obj['metadata.QuestionId'] = q
                question_id = q.split('_')[1]

                multiple_categories = row['Q12']
                if multiple_categories != 'No':
                    multiple_category_terms = multiple_categories.split(',')
                    multiple_category_terms = [cat.strip('Mood Term') for cat in multiple_category_terms]   
                else: 
                    multiple_category_terms = []    

                #check if the mood term is NaN
                
                json_obj['QX_X.UserProvidedMoodTerm'] = row[q]

                #clean the mood term
                #step 1 - remove leading hashtag
                mood_term = row[q].lstrip('#').strip()
                #step 2 - lowercase
                mood_term = mood_term.lower()
                #step 3 - spell check
                spell = SpellChecker()
                mood_term_cleaned = spell.correction(mood_term)

                if mood_term_cleaned != None:
                    json_obj['QX_X.CleanedMoodTerm'] = mood_term_cleaned
                else: json_obj['QX_X.CleanedMoodTerm'] = mood_term
                    
                #gather the user mapped category
                json_obj['QX_X.UserCategoryMapping'] = row[avoid_categories[id]]

                #identify multiple categories
                if question_id in multiple_category_terms:
                    json_obj['QX.MultipleCategoryTerm'] = True

                    if pd.notna(row[f'Q13#1_{int(question_id)+1}']):
                        #split the multiple categories by comma
                        json_obj['QX#1_X.MultipleCategoryMapping'] = row[f'Q13#1_{int(question_id)+1}'].split(',')
                    else: 
                        json_obj['QX#1_X.MultipleCategoryMapping'] = None

                    if pd.notna(row[f'Q13#2_{int(question_id)+1}_1']):
                        json_obj['QX#2_X_1.UserProvidedCategory'] = row[f'Q13#2_{int(question_id)+1}_1']
                        #clean the category
                        user_provided_category = row[f'Q13#2_{int(question_id)+1}_1'].strip().lower()
                        user_provided_category_cleaned = spell.correction(user_provided_category)

                        if mood_term_cleaned != None:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category_cleaned
                        else:
                            json_obj['QX#2_X_1.CleanedCategory'] = user_provided_category
                    else:
                        json_obj['QX#2_X_1.UserProvidedCategory'] = None
                        json_obj['QX#2_X_1.CleanedCategory'] = None

                else:
                    json_obj['QX.MultipleCategoryTerm'] = False
                    json_obj['QX#1_X.MultipleCategoryMapping'] = None
                    json_obj['QX#2_X_1.UserProvidedCategory'] = None
                    json_obj['QX#2_X_1.CleanedCategory'] = None

                avoid_q_terms.append(json_obj)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        break

In [ ]:
spell.correction('catagory')

In [ ]:
# convert the array to a DataFrame and save to CSV
avoid_q_df = pd.DataFrame(avoid_q_terms)
avoid_q_df.to_csv('Data/Outbound/avoid_q_terms.csv', index=False, header=True)


In [ ]:
avoid_q_df.shape

In [ ]:
#test to ensure that all terms are captured
avoid_terms_df = response_df[response_df['Progress'] >= 100][avoid_q].copy()
total_avoid_terms = pd.DataFrame(avoid_terms_df[avoid_q].count(axis=1))
total_avoid_terms.rename(columns={0:'Total_Avoid_Terms'}, inplace=True)
total_avoid_terms.sum()

## Merge all data frames

In [ ]:
term_df = pd.concat([recent_q_df, fav_q_df, avoid_q_df], ignore_index=True)
term_df.reset_index(drop=True, inplace=True)
term_df.to_csv('Data/Outbound/all_terms.csv', index=False, header=True)
term_df.to_excel('Data/Outbound/all_terms2.xlsx', index=False, header=True)